Update the notebook filename in the `href` attribute of the launch button below before publishing.

<a href="https://jupyterhub.user.eopf.eodc.eu/hub/user-redirect/git-pull?repo=https://github.com/YOUR_ORG/YOUR_REPO&branch=main&urlpath=lab/tree/YOUR_REPO/landslide_sar_detection.ipynb" target="_blank">
  <button style="background-color:#0072ce; color:white; padding:0.6em 1.2em; font-size:1rem; border:none; border-radius:6px; margin-top:1em;">
    &#9654; Launch this notebook in JupyterLab
  </button>
</a>

### Introduction

In this notebook, we demonstrate how to detect landslides using **Sentinel-1 SAR (Synthetic Aperture Radar) backscatter change analysis** with data accessed in cloud-native **Zarr format** via the EOPF Sample Service. The workflow covers the full pipeline -- from streaming Sentinel-1 GRD data stored as Zarr directly into memory, through speckle filtering and multi-temporal log-ratio change detection, to the delineation and mapping of individual landslide scars.

The case study focuses on **Bududa District, Eastern Uganda** -- a high-frequency landslide hotspot on the footslopes of Mt. Elgon in the Greater Horn of Africa -- using pre-event (April 2022) and post-event (May 2022) Sentinel-1 IW GRD acquisitions surrounding a rainfall-triggered mass movement event. The same workflow applies directly to any landslide-prone region accessible via Sentinel-1 Zarr archives on the Copernicus Data Space Ecosystem (CDSE).

### What we will learn

- **SAR backscatter physics** -- why landslide scars produce distinctive VV backscatter increases and VH decreases in C-band SAR imagery
- **Cloud-native Zarr data access** -- how to stream Sentinel-1 GRD data in Zarr format using `zarr`, `xarray`, and `Dask`, loading only the chunks needed rather than downloading entire scenes
- **SAR preprocessing** -- converting linear sigma-naught to dB scale and applying speckle filtering to reduce multiplicative radar noise
- **Multi-temporal change detection** -- computing the Log-ratio Change Index (LCI), z-score normalisation against a stable background, and CUSUM cumulative sum analysis across a time series
- **Landslide delineation & validation** -- applying Otsu adaptive thresholding and morphological post-processing, then evaluating detections with Precision, Recall, F1, IoU, and Cohen's Kappa
- **Zarr chunking strategy** -- understanding how `(1, 512, 512)` time-first chunk layouts optimise I/O for change detection workflows compared to full-spatial or spatiotemporal chunk designs

### Prerequisites

**Python packages used in this notebook:**

| Package | Purpose |
|---|---|
| [`zarr`](https://zarr.readthedocs.io) | Reading and writing cloud-native chunked array stores |
| [`xarray`](https://docs.xarray.dev) | Labelled N-D array operations with Zarr/Dask backends |
| [`dask`](https://docs.dask.org) | Parallel lazy computation over chunked arrays |
| [`rioxarray`](https://corteva.github.io/rioxarray) | CRS assignment and raster I/O via xarray |
| [`scipy`](https://docs.scipy.org) | Speckle filtering (`uniform_filter`) and statistics |
| [`scikit-image`](https://scikit-image.org) | Otsu thresholding, morphological ops, region properties |
| [`matplotlib`](https://matplotlib.org) | Scientific figure generation |
| [`cmocean`](https://matplotlib.org/cmocean) | Perceptually uniform colormaps for geoscience |

**Background knowledge recommended:**
- Basic familiarity with SAR remote sensing concepts (backscatter, polarisation) -- see [ESA SAR Educational Resources](https://earth.esa.int/eogateway/catalog/sentinel-1-sar-educational-resources)
- Understanding of `xarray` DataArrays and Datasets -- see the [xarray tutorial](https://tutorial.xarray.dev)
- Introduction to Zarr format -- see the [Zarr Getting Started guide](https://zarr.readthedocs.io/en/stable/getting_started.html)
- Familiarity with [Sentinel-1 IW GRD product specification](https://sentinel.esa.int/web/sentinel/technical-guides/sentinel-1-sar/products-algorithms/level-1-algorithms/ground-range-detected)

> **Note:** This notebook uses a synthetic in-memory Zarr store that mirrors the EOPF/CDSE real-data structure exactly. To work with live Sentinel-1 Zarr data, register at the [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu) and substitute the `s3fs` connection shown in Section 2.

#### Import libraries

In [ ]:
# Uncomment to install in a fresh environment (e.g. Google Colab)
# !pip install -q zarr==2.18.3 "xarray[complete]" "dask[distributed]" \
#              s3fs rioxarray rasterio geopandas scipy scikit-image folium cmocean tqdm

import warnings
warnings.filterwarnings('ignore')

import zarr                                          # Cloud-native chunked array storage
import s3fs                                          # S3-compatible filesystem for remote Zarr
import numpy as np                                   # Numerical array operations
import pandas as pd                                  # Tabular data and DataFrames
import xarray as xr                                  # Labelled N-D arrays with Zarr/Dask backends
import rioxarray                                     # Raster CRS and spatial ops for xarray
import dask
import dask.array as da                              # Lazy parallel array computation
from dask.distributed import Client, LocalCluster   # Dask multi-worker scheduler
from scipy.ndimage import uniform_filter             # Boxcar speckle filter
from scipy.stats import norm
from skimage.filters import threshold_otsu           # Adaptive Otsu thresholding
from skimage.morphology import (
    binary_opening, binary_closing, disk, remove_small_objects,
)  # Morphological operations
from skimage.measure import regionprops_table        # Patch statistics (area, centroid)
from scipy.ndimage import label
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import cmocean                                       # Perceptually uniform geoscience colormaps
from datetime import datetime
import pathlib

print(
    f"zarr {zarr.__version__} | xarray {xr.__version__} "
    f"| dask {dask.__version__} | numpy {np.__version__}"
)
print("All libraries imported successfully.")

#### Helper functions

##### `create_synthetic_sentinel1_zarr`

In [ ]:
def create_synthetic_sentinel1_zarr(bbox, dates, resolution_deg=0.00009, seed=42):
    """
    Build a synthetic in-memory Zarr store that mirrors the EOPF/CDSE
    Sentinel-1 GRD product structure, including injected landslide scars.

    Parameters
    ----------
    bbox : dict
        Bounding box with keys min_lon, max_lon, min_lat, max_lat.
    dates : list of str
        ISO date strings for each acquisition (e.g. "2022-04-20").
    resolution_deg : float
        Pixel size in degrees (~10 m at equator). Default 0.00009.
    seed : int
        Random seed for reproducibility. Default 42.

    Returns
    -------
    tuple
        (zarr.hierarchy.Group, np.ndarray) -- store and ground-truth scar mask.
    """
    rng  = np.random.default_rng(seed)
    lons = np.arange(bbox['min_lon'], bbox['max_lon'], resolution_deg)
    lats = np.arange(bbox['max_lat'], bbox['min_lat'], -resolution_deg)  # North to South
    T, Y, X = len(dates), len(lats), len(lons)

    # Base backscatter -- realistic forest/hillside terrain.
    # Mean sigma0 ~ -8 dB (linear ~0.14) for tropical forest in IW mode.
    base_vv = rng.lognormal(np.log(0.14), 0.6, (Y, X)).astype(np.float32)
    base_vh = rng.lognormal(np.log(0.04), 0.6, (Y, X)).astype(np.float32)

    # Three synthetic elliptical landslide scars
    yy, xx = np.ogrid[:Y, :X]
    combined_scar = np.zeros((Y, X), dtype=bool)
    for cy, cx, ry, rx in [
        (Y // 3,     X // 4,      60, 40),
        (Y // 2,     X // 2 + 20, 45, 30),
        (2 * Y // 3, X // 3,      35, 25),
    ]:
        combined_scar |= (
            (yy - cy) ** 2 / ry ** 2 + (xx - cx) ** 2 / rx ** 2
        ) < 1.0

    # Stack time-series; inject scar signal in post-event scenes.
    POST_DATE = dates[len(dates) // 2]   # first post-event acquisition date
    vv_stack  = np.zeros((T, Y, X), dtype=np.float32)
    vh_stack  = np.zeros((T, Y, X), dtype=np.float32)
    for t, d in enumerate(dates):
        vv = base_vv * rng.lognormal(0, 0.15, (Y, X)).astype(np.float32)
        vh = base_vh * rng.lognormal(0, 0.15, (Y, X)).astype(np.float32)
        if d >= POST_DATE:   # post-event: VV up, VH down over scar
            n = int(combined_scar.sum())
            vv[combined_scar] *= rng.uniform(2.5, 5.0, n).astype(np.float32)
            vh[combined_scar] *= rng.uniform(0.2, 0.5, n).astype(np.float32)
        vv_stack[t], vh_stack[t] = vv, vh

    # Assemble Zarr store with EOPF-compatible metadata
    store = zarr.group(store=zarr.MemoryStore())
    store.attrs.update({
        'title': 'Sentinel-1 GRD IW Sigma-Naught Backscatter',
        'sensor': 'Sentinel-1A/B',
        'mode': 'IW',
        'product': 'GRD-H',
        'polarizations': ['VV', 'VH'],
        'orbit_direction': 'DESCENDING',
        'crs': 'EPSG:4326',
        'convention': 'sigma0_linear',
        'zarr_format': 2,
        'source': 'Synthetic -- mirrors Copernicus Data Space Ecosystem structure',
        'created': datetime.utcnow().isoformat() + 'Z',
    })
    store.array('time', data=np.array(dates, dtype='U10'), dtype='U10')
    store.array('lat',  data=lats.astype(np.float64), dtype='f8')
    store.array('lon',  data=lons.astype(np.float64), dtype='f8')

    chunk      = (1, min(512, Y), min(512, X))   # time-first chunk layout
    compressor = zarr.Blosc(cname='zstd', clevel=5, shuffle=zarr.Blosc.BITSHUFFLE)
    store.array(
        'VV', data=vv_stack, chunks=chunk, dtype='f4', compressor=compressor,
        attrs={'long_name': 'VV sigma0 linear', 'units': '1'},
    )
    store.array(
        'VH', data=vh_stack, chunks=chunk, dtype='f4', compressor=compressor,
        attrs={'long_name': 'VH sigma0 linear', 'units': '1'},
    )
    store.array(
        'scar_mask', data=combined_scar.astype(np.uint8), dtype='u1',
        attrs={'description': 'Ground-truth landslide scar mask (1=scar)'},
    )
    return store, combined_scar

##### `cusum_change_score`

In [ ]:
def cusum_change_score(ts_array, k=0.5):
    """
    Apply bidirectional CUSUM (Cumulative Sum) change detection to a
    (T, Y, X) backscatter time-series array.

    The CUSUM statistic accumulates standardised deviations from the
    temporal mean. A rising score indicates a persistent shift in the
    signal -- the key signature of a landslide scar changing the surface.

        S_t = max(0, S_{t-1} + (x_t - mu) / sigma - k)

    Parameters
    ----------
    ts_array : np.ndarray, shape (T, Y, X)
        sigma0 time-series (dB or linear).
    k : float
        Allowable slack (typically 0.5--1.0). Default 0.5.

    Returns
    -------
    tuple of np.ndarray
        (cusum_pos, cusum_neg) -- final positive and negative cumulative
        scores, each of shape (Y, X).
    """
    T, Y, X    = ts_array.shape
    mu         = ts_array.mean(axis=0, keepdims=True)
    sig        = ts_array.std(axis=0,  keepdims=True) + 1e-6
    normalised = (ts_array - mu) / sig
    cusum_pos  = np.zeros_like(ts_array)
    cusum_neg  = np.zeros_like(ts_array)
    for t in range(1, T):
        cusum_pos[t] = np.maximum(0, cusum_pos[t - 1] + normalised[t] - k)
        cusum_neg[t] = np.maximum(0, cusum_neg[t - 1] - normalised[t] - k)
    return cusum_pos[-1], cusum_neg[-1]   # return final time-step scores

##### `compute_accuracy_metrics`

In [ ]:
def compute_accuracy_metrics(predicted, reference):
    """
    Compute standard binary classification accuracy metrics.

    Parameters
    ----------
    predicted : np.ndarray of bool
        Binary detection mask.
    reference : np.ndarray of bool
        Ground-truth reference mask, same shape as predicted.

    Returns
    -------
    dict
        Keys: TP, TN, FP, FN, Precision, Recall, F1-Score, IoU,
        Overall Accuracy, Cohen's Kappa.
    """
    TP    = int(( predicted &  reference).sum())
    TN    = int((~predicted & ~reference).sum())
    FP    = int(( predicted & ~reference).sum())
    FN    = int((~predicted &  reference).sum())
    total = TP + TN + FP + FN
    prec  = TP / (TP + FP + 1e-9)
    rec   = TP / (TP + FN + 1e-9)
    f1    = 2 * prec * rec / (prec + rec + 1e-9)
    iou   = TP / (TP + FP + FN + 1e-9)
    oa    = (TP + TN) / total
    pe    = (
        (TP + FP) * (TP + FN) + (FN + TN) * (FP + TN)
    ) / total ** 2
    kappa = (oa - pe) / (1 - pe + 1e-9)
    return {
        'TP': TP, 'TN': TN, 'FP': FP, 'FN': FN,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'IoU': iou,
        'Overall Accuracy': oa,
        "Cohen's Kappa": kappa,
    }

Three helper functions power the workflow:

- **`create_synthetic_sentinel1_zarr`** -- builds an in-memory Zarr store mirroring the EOPF/CDSE Sentinel-1 GRD structure with injected landslide scar signatures; swap this for an `s3fs.S3Map` connection to use live CDSE data.
- **`cusum_change_score`** -- applies bidirectional CUSUM temporal analysis across the sigma0 time-series to detect persistent signal shifts caused by landslide surface change.
- **`compute_accuracy_metrics`** -- evaluates binary detection performance with Precision, Recall, F1, IoU, and Cohen's Kappa against a ground-truth reference mask.

<hr>

## 1. SAR Backscatter Physics & Scientific Background

Before accessing any data, it is important to understand *why* SAR backscatter changes when a landslide occurs. SAR backscatter ($\sigma^0$, sigma-naught) in the C-band (5.4 GHz, ~5.6 cm wavelength) is primarily sensitive to **surface roughness**, **dielectric constant** (related to soil moisture), and **volume scattering** from vegetation canopies.

A landslide scar removes the forest canopy and exposes bare, disturbed soil or debris. This causes a characteristic two-polarisation response:

| Polarisation | Pre-event | Post-event | Reason |
|---|---|---|---|
| **VV** | Low (forest floor shadowed) | **Increases** | Surface scattering from rough debris |
| **VH** | High (canopy volume scattering) | **Decreases** | Loss of forest volume scatterers |

The standard metric for detecting this change is the **Log-ratio Change Index (LCI)**:

$$\text{LCI} = 10 \cdot \log_{10}\!\left(\frac{\sigma^0_{\text{post}}}{\sigma^0_{\text{pre}}}\right) = \sigma^0_{\text{post}}(\text{dB}) - \sigma^0_{\text{pre}}(\text{dB})$$

Large positive LCI in VV (bright scar) combined with negative LCI in VH (dark scar) provides strong, discriminative evidence of a landslide even under heavy cloud cover -- a critical advantage over optical sensors in the monsoon-affected Greater Horn of Africa.

In [ ]:
# Visualise the SAR backscatter change physics
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.patch.set_facecolor('#1a1a2e')
x = np.linspace(-15, 15, 400)

# Panel A: typical sigma0 distributions pre/post landslide
ax = axes[0]
ax.set_facecolor('#16213e')
ax.fill_between(x, norm.pdf(x, -8, 2.5), alpha=0.4, color='#4ecdc4', label='Pre-event sigma0')
ax.fill_between(x, norm.pdf(x, -4, 2.8), alpha=0.4, color='#ff6b6b', label='Post-event (landslide)')
ax.fill_between(x, norm.pdf(x, -8.2, 2.5), alpha=0.3, color='#a8b2d8', label='Post-event (stable)')
ax.set_xlabel('sigma0 (dB)', color='white')
ax.set_ylabel('Density', color='white')
ax.set_title('A. Backscatter Distributions', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#0f3460', labelcolor='white', fontsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

# Panel B: LCI histogram with detection threshold
ax = axes[1]
ax.set_facecolor('#16213e')
rng_vis = np.random.default_rng(0)
lci_vis = np.concatenate([
    rng_vis.normal(0, 1.2, 8000),
    rng_vis.normal(4.5, 1.8, 800),
])
ax.hist(lci_vis, bins=80, color='#4ecdc4', alpha=0.7, density=True)
ax.axvline(2.5,  color='#ff6b6b', ls='--', lw=2, label='Detection threshold')
ax.axvline(-2.5, color='#ff6b6b', ls='--', lw=2)
ax.fill_betweenx([0, 0.25], 2.5, 10, alpha=0.15, color='#ff6b6b', label='Detected change')
ax.set_xlabel('LCI (dB)', color='white')
ax.set_ylabel('Density', color='white')
ax.set_title('B. LCI Distribution & Threshold', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#0f3460', labelcolor='white', fontsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

# Panel C: Sentinel-1 IW acquisition geometry (schematic)
ax = axes[2]
ax.set_facecolor('#16213e')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.plot(2, 9, 's', color='#e94560', ms=12, label='Sentinel-1 satellite')
ax.add_patch(plt.Polygon([[1, 0], [3, 0], [5, 9], [2, 9]], alpha=0.15, color='#4ecdc4'))
theta_vals = np.linspace(90, 90 - 38.5, 30)
ax.plot(
    2 + 2 * np.cos(np.radians(theta_vals)),
    9 + 2 * np.sin(np.radians(theta_vals)),
    color='#ffd700', lw=2,
)
ax.annotate(u'θ=38.5°', xy=(3.2, 8.2), color='#ffd700', fontsize=9)
terrain_x = np.linspace(1, 5, 100)
terrain_y = 0.8 * np.sin(terrain_x) + 1.5
ax.fill_between(terrain_x, terrain_y, 0, alpha=0.4, color='#2d6a4f', label='Forest terrain')
scar_x = np.linspace(2.8, 3.8, 50)
ax.fill_between(
    scar_x, 0.8 * np.sin(scar_x) + 1.5, 0, alpha=0.7, color='#ff6b6b', label='Landslide scar',
)
ax.set_title('C. IW Acquisition Geometry', color='white')
ax.tick_params(colors='white')
ax.legend(facecolor='#0f3460', labelcolor='white', fontsize=8, loc='upper right')
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

plt.suptitle(
    'Sentinel-1 SAR Backscatter Change Physics for Landslide Detection',
    color='white', fontsize=13, y=1.02, fontweight='bold',
)
plt.tight_layout()
plt.savefig('fig01_sar_physics.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

### Study area: Bududa District, Uganda

Bududa District on the footslopes of Mt. Elgon (eastern Uganda) is one of the most landslide-prone areas in sub-Saharan Africa, with major events recorded in 2010, 2018, and 2022. The steep volcanic slopes, deep weathered soils, heavy seasonal rainfall (>1,400 mm/year), and dense agricultural smallholdings create ideal conditions for rainfall-triggered shallow translational landslides. This makes it an ideal case study for SAR-based detection workflows that can feed into ICPAC's operational early warning systems.

**Acquisition parameters used in this notebook:**

| Parameter | Value |
|---|---|
| Mode | IW (Interferometric Wide Swath) |
| Product | GRD-H (Ground Range Detected, High Resolution) |
| Polarisations | VV + VH |
| Orbit direction | Descending |
| Pre-event baseline | 2022-03-03 -- 2022-04-20 (4 scenes, mean) |
| Post-event scene | 2022-05-26 |
| Spatial resolution | ~10 m (after multilooking) |

In [ ]:
# Global constants
STUDY_BBOX = {
    'min_lon': 34.10,
    'max_lon': 34.45,
    'min_lat':  0.85,
    'max_lat':  1.15,
}  # Bududa District, Uganda

PRE_DATE  = '2022-04-20'
POST_DATE = '2022-05-26'
DATES = [
    '2022-03-03', '2022-03-15', '2022-04-08', '2022-04-20',
    '2022-05-02', '2022-05-14', '2022-05-26', '2022-06-07',
]

POLARIZATION      = 'VV'
SPECKLE_WINDOW    = 7     # pixels; boxcar filter window size
ZSCORE_THRESHOLD  = 2.5   # standard deviations above stable background
MIN_PATCH_SIZE_HA = 0.5   # minimum scar area to retain (hectares)
TARGET_RES        = 10    # metres per pixel
TARGET_CRS        = 'EPSG:32636'  # UTM Zone 36N

print("Study region : Bududa District, Eastern Uganda")
print(f"Pre-event    : {DATES[0]} to {DATES[3]}  "
      f"({sum(d <= PRE_DATE for d in DATES)} scenes)")
print(f"Post-event   : {POST_DATE}")
print(f"Polarisation : {POLARIZATION} + VH")

## 2. Data Access -- Sentinel-1 in Cloud-Native Zarr Format

Traditional workflows require downloading complete Sentinel-1 SAFE packages (often >1 GB per scene) before any processing can begin. The EOPF Sample Service exposes Sentinel-1 GRD data as **Zarr stores** on S3-compatible cloud storage, enabling partial reads -- we stream only the spatial subregion and time slices we need.

To connect to real EOPF/CDSE data, replace the `create_synthetic_sentinel1_zarr(...)` call below with:

```python
import s3fs
fs    = s3fs.S3FileSystem(anon=True, endpoint_url='https://eodata.dataspace.copernicus.eu')
store = zarr.open(
    s3fs.S3Map('eodata/Sentinel-1/GRD/2022/05/26/<product_id>/zarr/', fs=fs),
    mode='r',
)
```

The chunk layout used by the EOPF service is `(1, 512, 512)` -- one time-slice per chunk -- which means retrieving the pre- and post-event scenes requires reading exactly **2 chunks per band** regardless of the total archive length.

In [ ]:
# Build / connect to the Sentinel-1 Zarr store.
# In production, replace this call with an s3fs.S3Map connection to CDSE.
print("Building synthetic Sentinel-1 Zarr store (mirrors EOPF/CDSE structure)...")
zarr_store, gt_mask = create_synthetic_sentinel1_zarr(STUDY_BBOX, DATES)

# Inspect the store structure and metadata
print("\nZarr store tree:")
print(zarr_store.tree())

vv_arr = zarr_store['VV']
print(f"\n  Shape       : {vv_arr.shape}  (T x Y x X)")
print(f"  Chunks      : {vv_arr.chunks}  -- time-first layout")
print(f"  Compressor  : {vv_arr.compressor}")
cr = vv_arr.nbytes / max(vv_arr.nbytes_stored, 1)
print(f"  Compression : {cr:.1f}x reduction vs uncompressed float32")

### Load Zarr into xarray Dataset with lazy Dask arrays

In [ ]:
# Start a local Dask cluster for parallel chunk reads
cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='4GB')
client  = Client(cluster)
print(f"Dask dashboard: {client.dashboard_link}")

# Wrap Zarr arrays as a labelled xarray Dataset
lats  = zarr_store['lat'][:]
lons  = zarr_store['lon'][:]
times = pd.to_datetime(zarr_store['time'][:])

ds = xr.Dataset(
    {
        'VV': xr.DataArray(
            da.from_zarr(zarr_store['VV']),
            dims=['time', 'lat', 'lon'],
            coords={'time': times, 'lat': lats, 'lon': lons},
            attrs=zarr_store['VV'].attrs,
        ),
        'VH': xr.DataArray(
            da.from_zarr(zarr_store['VH']),
            dims=['time', 'lat', 'lon'],
            coords={'time': times, 'lat': lats, 'lon': lons},
            attrs=zarr_store['VH'].attrs,
        ),
    },
    attrs=dict(zarr_store.attrs),
)
ds = ds.rio.write_crs('EPSG:4326')
print(ds)
print(f"\nLoaded as lazy Dask arrays -- no data read yet  ({ds.nbytes / 1e6:.1f} MB logical)")

## 3. SAR Preprocessing

Raw Sentinel-1 GRD data is stored as sigma-naught in **linear power scale** (values typically 0.001--1.0). Two preprocessing steps are required before change detection:

1. **Linear to dB conversion**: $\sigma^0_{\text{dB}} = 10 \cdot \log_{10}(\sigma^0_{\text{linear}})$ -- transforms the skewed log-normal distribution into approximately Gaussian, enabling z-score statistics.
2. **Speckle filtering**: SAR images contain multiplicative speckle noise. A boxcar (uniform) filter in the dB domain suppresses this noise while preserving spatial structures larger than the window size.

In a full operational pipeline, **Range-Doppler Terrain Correction (RTC)** would also be applied using a DEM (e.g., Copernicus DEM GLO-30) to remove geometric distortions caused by steep terrain -- especially important in the Mt. Elgon study area.

In [ ]:
# Step 1: Convert linear sigma0 to dB scale
EPSILON = 1e-10    # prevent log(0) for near-zero pixels

ds_db = xr.Dataset(attrs=ds.attrs)
for pol in ['VV', 'VH']:
    ds_db[pol] = 10.0 * xr.apply_ufunc(
        np.log10, ds[pol].clip(min=EPSILON),
        dask='parallelized', output_dtypes=[np.float32],
    )
    ds_db[pol].attrs.update({'units': 'dB', 'long_name': f'{pol} sigma0 (dB)'})

ds_db = ds_db.compute()    # trigger Dask graph and load into memory
print(f"sigma0 VV : {float(ds_db.VV.min()):.1f} to {float(ds_db.VV.max()):.1f} dB")
print(f"sigma0 VH : {float(ds_db.VH.min()):.1f} to {float(ds_db.VH.max()):.1f} dB")

# Step 2: Speckle filter -- boxcar in dB domain
ds_filt = xr.Dataset(attrs=ds_db.attrs)
for pol in ['VV', 'VH']:
    print(f"Speckle filtering {pol} ({SPECKLE_WINDOW}x{SPECKLE_WINDOW} boxcar)...", end=' ')
    filtered = uniform_filter(
        ds_db[pol].values.astype(np.float32),
        size=(1, SPECKLE_WINDOW, SPECKLE_WINDOW),
        mode='reflect',
    )
    ds_filt[pol] = xr.DataArray(
        filtered,
        dims=ds_db[pol].dims,
        coords=ds_db[pol].coords,
        attrs={**ds_db[pol].attrs,
               'speckle_filter': f'Boxcar {SPECKLE_WINDOW}x{SPECKLE_WINDOW}'},
    )
    print('done')
print("\nPreprocessing complete.")

In [ ]:
t_pre = 3   # index of the last pre-event scene (2022-04-20)
ext   = [
    STUDY_BBOX['min_lon'], STUDY_BBOX['max_lon'],
    STUDY_BBOX['min_lat'], STUDY_BBOX['max_lat'],
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#1a1a2e')

panels = [
    ('Raw sigma0 VV (pre)',      ds_db.VV[t_pre]),
    ('Filtered sigma0 VV (pre)', ds_filt.VV[t_pre]),
    ('Raw sigma0 VH (pre)',      ds_db.VH[t_pre]),
    ('Filtered sigma0 VH (pre)', ds_filt.VH[t_pre]),
]

for ax, (title, data) in zip(axes.flat, panels):
    ax.set_facecolor('#16213e')
    im = ax.imshow(
        data.values, cmap=cmocean.cm.gray, vmin=-20, vmax=0,
        extent=ext, origin='upper', aspect='auto',
    )
    cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('sigma0 (dB)', color='white')
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.yaxis.get_ticklabels(), color='white')
    ax.set_title(title, color='white', pad=8)
    ax.set_xlabel('Longitude', color='white')
    ax.set_ylabel('Latitude', color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('gray')

plt.suptitle(
    f'Sentinel-1 SAR Preprocessing -- Pre-event Scene {DATES[t_pre]}\n'
    'Bududa District, Uganda',
    color='white', fontsize=13, y=1.01,
)
plt.tight_layout()
plt.savefig('fig02_raw_vs_filtered.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print("The speckle filter reduces the grainy texture while preserving the spatial structure of terrain features.")

## 4. Backscatter Change Detection

With preprocessed sigma0 time-series in hand, we compute change between the multi-temporal pre-event baseline and the post-event scene using three complementary approaches:

1. **Log-ratio Change Index (LCI)** -- the per-pixel dB difference between post and pre-event scenes. Using the mean of all pre-event scenes as the baseline suppresses orbital variability and seasonal effects.
2. **Z-score normalisation** -- standardises the LCI against the stable background (lower 60th percentile of |LCI|), making the threshold interpretation statistically meaningful regardless of scene-level radiometric variation.
3. **CUSUM temporal analysis** -- accumulates standardised deviations over all 8 time steps to detect the timing of persistent change, adding temporal evidence beyond the single pre/post comparison.

In [ ]:
# Separate pre-event and post-event stacks
pre_mask  = ds_filt.time < np.datetime64(POST_DATE)
post_mask = ds_filt.time >= np.datetime64(POST_DATE)

# Multi-temporal baseline: mean over all pre-event scenes
baseline_vv = ds_filt.VV.sel(time=pre_mask).mean(dim='time')
baseline_vh = ds_filt.VH.sel(time=pre_mask).mean(dim='time')

# Most recent post-event scene
post_vv = ds_filt.VV.sel(time=post_mask).isel(time=0)
post_vh = ds_filt.VH.sel(time=post_mask).isel(time=0)

# Log-ratio Change Index (dB difference = log ratio in the dB domain)
lci_vv       = (post_vv - baseline_vv).rename('LCI_VV')
lci_vh       = (post_vh - baseline_vh).rename('LCI_VH')
# Euclidean magnitude across polarisations
lci_combined = np.sqrt(lci_vv ** 2 + lci_vh ** 2).rename('LCI_combined')

# Z-score normalisation against stable background
stable_mask = (
    np.abs(lci_combined.values)
    < np.percentile(np.abs(lci_combined.values), 60)
)
mu_s  = float(lci_combined.values[stable_mask].mean())
sig_s = float(lci_combined.values[stable_mask].std()) + 1e-6
z_score = (lci_combined - mu_s) / sig_s

# CUSUM temporal analysis
cusum_pos, cusum_neg = cusum_change_score(ds_filt.VV.values.astype(np.float32))
cusum_score = cusum_pos + cusum_neg   # bidirectional combined score

print(f"LCI VV  | Mean: {float(lci_vv.mean()):+.2f} dB | Std: {float(lci_vv.std()):.2f} dB")
print(f"LCI VH  | Mean: {float(lci_vh.mean()):+.2f} dB | Std: {float(lci_vh.std()):.2f} dB")
print(f"Z-score | Range: {float(z_score.min()):.1f} to {float(z_score.max()):.1f} sigma")
print(f"CUSUM   | Range: {cusum_score.min():.2f} to {cusum_score.max():.2f}")

In [ ]:
# Normalise each component to [0, 1] and combine with weighted average
ALPHA  = 0.65   # weight for z-score evidence (remainder goes to CUSUM)

z_norm = np.abs(z_score.values)
z_norm = (z_norm - z_norm.min()) / (z_norm.max() - z_norm.min() + 1e-6)

c_norm = cusum_score.copy()
c_norm = (c_norm - c_norm.min()) / (c_norm.max() - c_norm.min() + 1e-6)

evidence = ALPHA * z_norm + (1.0 - ALPHA) * c_norm

print(f"Evidence image range : {evidence.min():.4f} to {evidence.max():.4f}")
print(f"Pixels above 0.5     : {(evidence > 0.5).sum():,} ({100 * (evidence > 0.5).mean():.1f}%)")

## 5. Landslide Detection & Delineation

The combined evidence image is converted to a binary landslide mask using **Otsu's method** -- an adaptive threshold that maximises inter-class variance without requiring manual parameter tuning. The binary mask is then refined with morphological operations to remove speckle-related false positives and fill small holes within real scars.

In [ ]:
# Otsu adaptive threshold
otsu_thresh = threshold_otsu(evidence)
# Enforce a minimum threshold of 0.55 to guard against low-contrast scenes
binary_raw  = evidence > max(otsu_thresh, 0.55)

# Morphological post-processing
# disk(3) at 10 m/pixel gives a ~30 m structuring element
selem        = disk(3)
binary_clean = binary_opening(binary_raw, footprint=selem)   # remove isolated noise
binary_clean = binary_closing(binary_clean, footprint=selem) # fill small internal holes

# Remove patches smaller than MIN_PATCH_SIZE_HA.
# 1 ha = 10,000 m^2; at TARGET_RES m/pixel, 1 pixel covers TARGET_RES^2 m^2.
# e.g. 0.5 ha / (10 m)^2 = 5000 m^2 / 100 m^2 = 50 pixels.
min_px       = int(MIN_PATCH_SIZE_HA * 1e4 / TARGET_RES ** 2)
binary_final = remove_small_objects(binary_clean, min_size=min_px)

# Connected-component labelling
labelled, n_scars = label(binary_final)

# Per-patch statistics
props = regionprops_table(
    labelled,
    intensity_image=lci_vv.values,
    properties=['label', 'area', 'centroid', 'eccentricity',
                'mean_intensity', 'max_intensity'],
)
df_patches = pd.DataFrame(props)
df_patches['area_ha']      = df_patches['area'] * TARGET_RES ** 2 / 1e4
df_patches['centroid_lat'] = lats[
    df_patches['centroid-0'].astype(int).clip(0, len(lats) - 1)
]
df_patches['centroid_lon'] = lons[
    df_patches['centroid-1'].astype(int).clip(0, len(lons) - 1)
]
df_patches['mean_lci_db'] = df_patches['mean_intensity'].round(2)
df_patches['max_lci_db']  = df_patches['max_intensity'].round(2)

print(f"Detected landslide patches : {n_scars}")
print(f"Total scar area            : {binary_final.sum() * TARGET_RES ** 2 / 1e4:.1f} ha")
print(f"Minimum patch retained     : {min_px} pixels ({MIN_PATCH_SIZE_HA} ha)")
print()
print(f"{'Label':>6} {'Area (ha)':>10} {'Lat':>9} {'Lon':>9} {'Mean LCI':>10} {'Max LCI':>9}")
print("-" * 58)
for _, r in df_patches.iterrows():
    print(
        f"{int(r['label']):>6} {r['area_ha']:>10.2f} "
        f"{r['centroid_lat']:>9.4f} {r['centroid_lon']:>9.4f} "
        f"{r['mean_lci_db']:>10.2f} {r['max_lci_db']:>9.2f}"
    )

## 6. Validation

In [ ]:
# Compute accuracy metrics against the synthetic ground-truth scar mask
gt_array = zarr_store['scar_mask'][:].astype(bool)
metrics  = compute_accuracy_metrics(binary_final, gt_array)

print("=" * 48)
print("  VALIDATION METRICS")
print("=" * 48)
for k, v in metrics.items():
    if k in ('TP', 'TN', 'FP', 'FN'):
        print(f"  {k:22s}: {v:,}")
    else:
        print(f"  {k:22s}: {v:.4f}")
print("=" * 48)

f1    = metrics['F1-Score']
grade = 'Excellent' if f1 > 0.8 else 'Good' if f1 > 0.65 else 'Moderate'
print(f"\nDetection quality : {grade}  (F1 = {f1:.3f})")

## Now it is your turn

The following exercises will help you deepen your understanding of SAR change detection and cloud-native Zarr data workflows. Each task is independent and can be completed by modifying the code cells above.

### Task 1: Change the Study Area
Landslides occur across the entire Greater Horn of Africa and beyond. Select a different high-risk region and repeat the workflow.

- Go to [bboxfinder.com](http://bboxfinder.com/) and draw a bounding box over a known landslide hotspot (e.g. the Rwenzori Mountains, Uganda/DRC border, or the Ethiopian Highlands).
- Update the `STUDY_BBOX` dictionary in the constants cell with your new coordinates.
- Re-run the notebook from Section 2 onwards. How does the spatial pattern of LCI change?

### Task 2: Explore Different Polarisations
This notebook primarily uses VV, but VH often carries a complementary signal.

- Change `POLARIZATION = 'VV'` to `'VH'` and recompute the LCI and z-score.
- Produce the detection figure for VH instead of VV.
- Compare the detection masks: where do VV and VH agree? Where do they disagree? What does this tell you about the scattering mechanism?

### Task 3: Tune the Detection Threshold
The Otsu threshold is fully adaptive, but you can also set it manually to explore the precision/recall trade-off.

- In the detection cell, replace `evidence > max(otsu_thresh, 0.55)` with a fixed threshold, e.g. `evidence > 0.45` and `evidence > 0.70`.
- For each threshold, call `compute_accuracy_metrics()` and record Precision, Recall, and F1.
- Plot a simple Precision-Recall curve. At what threshold is F1 maximised?

### Task 4: Investigate Zarr Chunking Performance
Chunking strategy controls how efficiently data is read from cloud storage.

- Modify `create_synthetic_sentinel1_zarr` to use a different chunk layout, e.g. `chunk = (8, 128, 128)` (temporal stacking) or `chunk = (8, 512, 512)` (full spatial).
- Print `vv_arr.chunks` and `vv_arr.nbytes_stored` for each layout.
- How does the number of chunks change? Which layout would be most efficient if you needed to access *all time steps* at a *single pixel* location?

### Task 5 (Advanced): Connect to Real CDSE Zarr Data
If you have a Copernicus Data Space Ecosystem account, try replacing the synthetic store with live Sentinel-1 Zarr data:

```python
import s3fs
fs = s3fs.S3FileSystem(
    key='YOUR_ACCESS_KEY',
    secret='YOUR_SECRET_KEY',
    endpoint_url='https://eodata.dataspace.copernicus.eu',
)
store = zarr.open(
    s3fs.S3Map('eodata/Sentinel-1/GRD/2022/05/26/<product_id>/zarr/', fs=fs),
    mode='r',
)
```

- Browse available Sentinel-1 products at [CDSE STAC Browser](https://browser.dataspace.copernicus.eu).
- Find a pre/post pair around a documented landslide event from the [NASA COOLR catalog](https://coolr.landslidedb.com).
- Run the full pipeline and compare your detected scars against the COOLR inventory.

## Conclusion

### Landslide Detection Pipeline Using Sentinel-1 SAR

In this notebook, we developed a **complete and reproducible landslide detection pipeline** using **Sentinel-1 SAR backscatter change analysis**, with data accessed in **cloud-native Zarr format** via the **EOPF Sample Service**.

### Key Insights

- **Efficient Data Access with Zarr** -- the time-first chunk layout `(1, 512, 512)` allows retrieval of only **two chunks per band** from a multi-year archive, significantly reducing I/O compared to traditional file-based workflows.

- **Robust SAR Backscatter Signal** -- SAR backscatter change (VV increases, VH decreases over landslide scars) provides a reliable **all-weather, day-night signal** for detecting mass movements, often within **six days of an event**.

- **Multi-Method Change Detection** -- the workflow integrates the Log-ratio Change Index (LCI), z-score normalisation, and CUSUM temporal analysis, fused into a single evidence image and refined with Otsu adaptive thresholding and morphological post-processing.

- **Operational Deployment Potential** -- the entire workflow can run in a standard Python environment without large data downloads, making it suitable for **operational deployment at ICPAC and partner NMHSs across the Greater Horn of Africa**.

---

### Detection Performance

| Metric | Value |
|---|---|
| Detected patches | See cell output above |
| Precision | See cell output above |
| Recall | See cell output above |
| F1-Score | See cell output above |
| IoU | See cell output above |

---

> **Limitation:** This notebook uses a synthetic Zarr store. For operational deployment, Sentinel-1 data should undergo **Range-Doppler Terrain Correction (RTC)** using the Copernicus DEM GLO-30 prior to change analysis, particularly in steep terrain such as the Mt. Elgon foothills.

Now that you can detect landslide scars from SAR backscatter change, the natural next step is to **combine SAR-derived scar maps with Sentinel-2 optical imagery** (available in the same Zarr format on CDSE) to characterise the vegetation recovery trajectory over the scar area in the months following an event -- a key input for displacement risk and long-term early warning assessment. See the [EOPF-101 Sentinel-2 Vegetation Change chapter](https://eopf.copernicus.eu/eopf-101) for a guided introduction to optical time-series analysis in Zarr format.

For the operational landslide early warning context, the methodology presented here feeds directly into ICPAC's **Multi-Sensor Landslide Early Warning System** pipeline:

> Sentinel-1 SAR scar detection (this notebook) &rarr; CHIRPS rainfall trigger &rarr; MCDA susceptibility filtering &rarr; HUSIKA mobile alert delivery &rarr; ReliefWeb inventory validation